Dynamic quantization gave the strongest lightweight improvement:
around 90% model size reduction with almost no loss in F1-score.

50% magnitude pruning also reduced model size:
NSL-KDD: 0.489 MB -> 0.184 MB
UNSW-NB15: 0.536 MB -> 0.199 MB



## Summary

Magnitude-based pruning was applied to the baseline CNN models by removing 50% of low-magnitude weights. The pruned NSL-KDD model achieved similar performance to the baseline with a smaller model size. The pruned UNSW-NB15 model slightly improved F1-score while reducing model size. These results show that pruning can reduce model complexity while maintaining acceptable intrusion detection performance.

In [18]:
import os

files_to_check = [
    "../models/nslkdd_cnn_pruned_model.h5",
    "../models/unsw_cnn_pruned_model.h5",
    "../results/pruned_model_evaluation_results.csv",
    "../results/nslkdd_pruned_confusion_matrix.csv",
    "../results/unsw_pruned_confusion_matrix.csv",
    "../results/all_model_comparison_results.csv"
]

for file in files_to_check:
    print(file, os.path.exists(file))

../models/nslkdd_cnn_pruned_model.h5 True
../models/unsw_cnn_pruned_model.h5 True
../results/pruned_model_evaluation_results.csv True
../results/nslkdd_pruned_confusion_matrix.csv True
../results/unsw_pruned_confusion_matrix.csv True
../results/all_model_comparison_results.csv True


In [17]:
nslkdd_pruned_cm_path = RESULTS_DIR / "nslkdd_pruned_confusion_matrix.csv"
unsw_pruned_cm_path = RESULTS_DIR / "unsw_pruned_confusion_matrix.csv"

pd.DataFrame(
    nslkdd_pruned_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
).to_csv(nslkdd_pruned_cm_path)

pd.DataFrame(
    unsw_pruned_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
).to_csv(unsw_pruned_cm_path)

print("NSL-KDD pruned confusion matrix saved:", nslkdd_pruned_cm_path.exists())
print("UNSW-NB15 pruned confusion matrix saved:", unsw_pruned_cm_path.exists())

NSL-KDD pruned confusion matrix saved: True
UNSW-NB15 pruned confusion matrix saved: True


The experimental results show that dynamic range quantization provided the strongest reduction in model size and inference time while maintaining almost identical detection performance. For NSL-KDD, the model size decreased from 0.489 MB to 0.046 MB, while the F1-score remained almost unchanged at approximately 74.95%. For UNSW-NB15, the model size decreased from 0.536 MB to 0.050 MB, while the F1-score remained approximately 94.18%.

Magnitude-based pruning with 50% sparsity also reduced the model size compared with the baseline models. For NSL-KDD, pruning slightly reduced the F1-score from 74.94% to 74.71%. For UNSW-NB15, pruning improved the F1-score from 94.18% to 94.69%, suggesting that some low-magnitude weights were not essential and may have contributed to model complexity without improving generalisation.

In [15]:
thesis_results = all_results.copy()

for col in ["Accuracy", "Precision", "Recall", "F1-score"]:
    thesis_results[col] = (thesis_results[col] * 100).round(2)

if "Training Time Seconds" in thesis_results.columns:
    thesis_results["Training Time Seconds"] = thesis_results["Training Time Seconds"].round(2)

if "Sparsity" in thesis_results.columns:
    thesis_results["Sparsity"] = (thesis_results["Sparsity"] * 100).round(2)

thesis_results["Inference Time Seconds"] = thesis_results["Inference Time Seconds"].round(3)
thesis_results["Model Size MB"] = thesis_results["Model Size MB"].round(3)

thesis_results

,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB,Sparsity
0,NSL-KDD,Baseline CNN,76.61,96.02,61.45,74.94,137.07,2.750,0.489,NaN
1,UNSW-NB15,Baseline CNN,92.23,96.10,92.33,94.18,43.73,14.661,0.536,NaN
2,NSL-KDD,Dynamic Quantized TFLite CNN,76.62,96.04,61.46,74.95,NaN,0.579,0.046,NaN
3,UNSW-NB15,Dynamic Quantized TFLite CNN,92.23,96.10,92.33,94.18,NaN,3.677,0.050,NaN
4,NSL-KDD,50% Magnitude Pruned CNN,76.48,96.26,61.05,74.71,NaN,2.601,0.184,50.0
5,UNSW-NB15,50% Magnitude Pruned CNN,92.82,95.47,93.92,94.69,NaN,14.605,0.199,50.0


In [14]:
BASELINE_RESULTS_PATH = RESULTS_DIR / "baseline_cnn_comparison.csv"
QUANTIZED_RESULTS_PATH = RESULTS_DIR / "quantized_tflite_evaluation_results.csv"
PRUNED_RESULTS_PATH = RESULTS_DIR / "pruned_model_evaluation_results.csv"

FINAL_ALL_RESULTS_PATH = RESULTS_DIR / "all_model_comparison_results.csv"

baseline_results = pd.read_csv(BASELINE_RESULTS_PATH)
quantized_results = pd.read_csv(QUANTIZED_RESULTS_PATH)
pruned_results = pd.read_csv(PRUNED_RESULTS_PATH)

all_results = pd.concat(
    [baseline_results, quantized_results, pruned_results],
    ignore_index=True
)

all_results.to_csv(FINAL_ALL_RESULTS_PATH, index=False)

print("All model comparison saved:", FINAL_ALL_RESULTS_PATH.exists())
all_results

All model comparison saved: True


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB,Sparsity
0,NSL-KDD,Baseline CNN,0.766057,0.960185,0.614509,0.749406,137.072274,2.750228,0.488823,NaN
1,UNSW-NB15,Baseline CNN,0.922289,0.961005,0.923287,0.941769,43.730621,14.660965,0.535698,NaN
2,NSL-KDD,Dynamic Quantized TFLite CNN,0.766191,0.960424,0.614587,0.749537,NaN,0.578795,0.046349,NaN
3,UNSW-NB15,Dynamic Quantized TFLite CNN,0.922300,0.961038,0.923270,0.941776,NaN,3.677338,0.050255,NaN
4,NSL-KDD,50% Magnitude Pruned CNN,0.764771,0.962644,0.610457,0.747127,NaN,2.601195,0.183807,0.5
5,UNSW-NB15,50% Magnitude Pruned CNN,0.928243,0.954661,0.939174,0.946854,NaN,14.604850,0.199432,0.5


In [13]:
def get_file_size_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)

pruning_results = pd.DataFrame([
    {
        "Dataset": "NSL-KDD",
        "Model": "50% Magnitude Pruned CNN",
        "Accuracy": nslkdd_pruned_accuracy,
        "Precision": nslkdd_pruned_precision,
        "Recall": nslkdd_pruned_recall,
        "F1-score": nslkdd_pruned_f1,
        "Sparsity": nslkdd_sparsity,
        "Inference Time Seconds": nslkdd_pruned_inference_time,
        "Model Size MB": get_file_size_mb(NSLKDD_PRUNED_MODEL_PATH)
    },
    {
        "Dataset": "UNSW-NB15",
        "Model": "50% Magnitude Pruned CNN",
        "Accuracy": unsw_pruned_accuracy,
        "Precision": unsw_pruned_precision,
        "Recall": unsw_pruned_recall,
        "F1-score": unsw_pruned_f1,
        "Sparsity": unsw_sparsity,
        "Inference Time Seconds": unsw_pruned_inference_time,
        "Model Size MB": get_file_size_mb(UNSW_PRUNED_MODEL_PATH)
    }
])

pruning_results.to_csv(PRUNING_RESULTS_PATH, index=False)

print("Pruning results saved:", PRUNING_RESULTS_PATH.exists())
pruning_results

Pruning results saved: True


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Sparsity,Inference Time Seconds,Model Size MB
0,NSL-KDD,50% Magnitude Pruned CNN,0.764771,0.962644,0.610457,0.747127,0.5,2.601195,0.183807
1,UNSW-NB15,50% Magnitude Pruned CNN,0.928243,0.954661,0.939174,0.946854,0.5,14.604850,0.199432


In [12]:
def get_file_size_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)

pruning_results = pd.DataFrame([
    {
        "Dataset": "NSL-KDD",
        "Model": "50% Magnitude Pruned CNN",
        "Accuracy": nslkdd_pruned_accuracy,
        "Precision": nslkdd_pruned_precision,
        "Recall": nslkdd_pruned_recall,
        "F1-score": nslkdd_pruned_f1,
        "Sparsity": nslkdd_sparsity,
        "Inference Time Seconds": nslkdd_pruned_inference_time,
        "Model Size MB": get_file_size_mb(NSLKDD_PRUNED_MODEL_PATH)
    },
    {
        "Dataset": "UNSW-NB15",
        "Model": "50% Magnitude Pruned CNN",
        "Accuracy": unsw_pruned_accuracy,
        "Precision": unsw_pruned_precision,
        "Recall": unsw_pruned_recall,
        "F1-score": unsw_pruned_f1,
        "Sparsity": unsw_sparsity,
        "Inference Time Seconds": unsw_pruned_inference_time,
        "Model Size MB": get_file_size_mb(UNSW_PRUNED_MODEL_PATH)
    }
])

pruning_results.to_csv(PRUNING_RESULTS_PATH, index=False)

print("Pruning results saved:", PRUNING_RESULTS_PATH.exists())
pruning_results

Pruning results saved: True


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Sparsity,Inference Time Seconds,Model Size MB
0,NSL-KDD,50% Magnitude Pruned CNN,0.764771,0.962644,0.610457,0.747127,0.5,2.601195,0.183807
1,UNSW-NB15,50% Magnitude Pruned CNN,0.928243,0.954661,0.939174,0.946854,0.5,14.604850,0.199432


In [11]:
nslkdd_pruned_model.save(NSLKDD_PRUNED_MODEL_PATH)
unsw_pruned_model.save(UNSW_PRUNED_MODEL_PATH)

print("NSL-KDD pruned model saved:", NSLKDD_PRUNED_MODEL_PATH.exists())
print("UNSW-NB15 pruned model saved:", UNSW_PRUNED_MODEL_PATH.exists())

NSL-KDD pruned model saved: True
UNSW-NB15 pruned model saved: True


In [10]:
start_time = time.time()

unsw_pruned_pred_prob = unsw_pruned_model.predict(X_unsw_test_cnn, verbose=0)
unsw_pruned_inference_time = time.time() - start_time

unsw_pruned_pred = (unsw_pruned_pred_prob >= 0.5).astype(int).ravel()

unsw_pruned_accuracy = accuracy_score(y_unsw_test, unsw_pruned_pred)
unsw_pruned_precision = precision_score(y_unsw_test, unsw_pruned_pred, zero_division=0)
unsw_pruned_recall = recall_score(y_unsw_test, unsw_pruned_pred, zero_division=0)
unsw_pruned_f1 = f1_score(y_unsw_test, unsw_pruned_pred, zero_division=0)

print("UNSW-NB15 Pruned CNN Results")
print(f"Accuracy: {unsw_pruned_accuracy:.6f}")
print(f"Precision: {unsw_pruned_precision:.6f}")
print(f"Recall: {unsw_pruned_recall:.6f}")
print(f"F1-score: {unsw_pruned_f1:.6f}")
print(f"Inference time: {unsw_pruned_inference_time:.6f} seconds")

unsw_pruned_cm = confusion_matrix(y_unsw_test, unsw_pruned_pred)

pd.DataFrame(
    unsw_pruned_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

UNSW-NB15 Pruned CNN Results
Accuracy: 0.928243
Precision: 0.954661
Recall: 0.939174
F1-score: 0.946854
Inference time: 14.604850 seconds


,Predicted Normal,Predicted Attack
Actual Normal,50677,5323
Actual Attack,7259,112082


In [9]:
unsw_baseline_model = tf.keras.models.load_model(UNSW_MODEL_PATH)

unsw_pruned_model, unsw_pruning_threshold, unsw_sparsity = prune_model_by_magnitude(
    unsw_baseline_model,
    pruning_percent=50
)

print("UNSW-NB15 pruning threshold:", unsw_pruning_threshold)
print("UNSW-NB15 model sparsity:", unsw_sparsity)
print("UNSW-NB15 sparsity percent:", unsw_sparsity * 100)

UNSW-NB15 pruning threshold: 0.060615562
UNSW-NB15 model sparsity: 0.5
UNSW-NB15 sparsity percent: 50.0


In [8]:
start_time = time.time()

nslkdd_pruned_pred_prob = nslkdd_pruned_model.predict(X_nslkdd_test_cnn, verbose=0)
nslkdd_pruned_inference_time = time.time() - start_time

nslkdd_pruned_pred = (nslkdd_pruned_pred_prob >= 0.5).astype(int).ravel()

nslkdd_pruned_accuracy = accuracy_score(y_nslkdd_test, nslkdd_pruned_pred)
nslkdd_pruned_precision = precision_score(y_nslkdd_test, nslkdd_pruned_pred, zero_division=0)
nslkdd_pruned_recall = recall_score(y_nslkdd_test, nslkdd_pruned_pred, zero_division=0)
nslkdd_pruned_f1 = f1_score(y_nslkdd_test, nslkdd_pruned_pred, zero_division=0)

print("NSL-KDD Pruned CNN Results")
print(f"Accuracy: {nslkdd_pruned_accuracy:.6f}")
print(f"Precision: {nslkdd_pruned_precision:.6f}")
print(f"Recall: {nslkdd_pruned_recall:.6f}")
print(f"F1-score: {nslkdd_pruned_f1:.6f}")
print(f"Inference time: {nslkdd_pruned_inference_time:.6f} seconds")

nslkdd_pruned_cm = confusion_matrix(y_nslkdd_test, nslkdd_pruned_pred)

pd.DataFrame(
    nslkdd_pruned_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

NSL-KDD Pruned CNN Results
Accuracy: 0.764771
Precision: 0.962644
Recall: 0.610457
F1-score: 0.747127
Inference time: 2.601195 seconds


,Predicted Normal,Predicted Attack
Actual Normal,9407,304
Actual Attack,4999,7834


In [7]:
nslkdd_baseline_model = tf.keras.models.load_model(NSLKDD_MODEL_PATH)

nslkdd_pruned_model, nslkdd_pruning_threshold, nslkdd_sparsity = prune_model_by_magnitude(
    nslkdd_baseline_model,
    pruning_percent=50
)

print("NSL-KDD pruning threshold:", nslkdd_pruning_threshold)
print("NSL-KDD model sparsity:", nslkdd_sparsity)
print("NSL-KDD sparsity percent:", nslkdd_sparsity * 100)

NSL-KDD pruning threshold: 0.09023816
NSL-KDD model sparsity: 0.5
NSL-KDD sparsity percent: 50.0


In [6]:
def prune_model_by_magnitude(model, pruning_percent=50):
    pruned_model = tf.keras.models.clone_model(model)
    pruned_model.set_weights(model.get_weights())

    all_weights = []

    for layer in pruned_model.layers:
        weights = layer.get_weights()
        for weight_array in weights:
            if weight_array.ndim > 1:
                all_weights.append(np.abs(weight_array).flatten())

    all_weights = np.concatenate(all_weights)
    threshold = np.percentile(all_weights, pruning_percent)

    total_weights = 0
    zeroed_weights = 0

    for layer in pruned_model.layers:
        weights = layer.get_weights()
        new_weights = []

        for weight_array in weights:
            if weight_array.ndim > 1:
                mask = np.abs(weight_array) >= threshold
                pruned_array = weight_array * mask

                total_weights += weight_array.size
                zeroed_weights += np.sum(pruned_array == 0)

                new_weights.append(pruned_array)
            else:
                new_weights.append(weight_array)

        layer.set_weights(new_weights)

    pruned_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    sparsity = zeroed_weights / total_weights

    return pruned_model, threshold, sparsity

In [4]:
nslkdd_test_df = pd.read_csv(DATA_DIR / "nslkdd_test_processed.csv")
unsw_test_df = pd.read_csv(DATA_DIR / "unsw_test_processed.csv")

nslkdd_scaler = joblib.load(NSLKDD_SCALER_PATH)
unsw_scaler = joblib.load(UNSW_SCALER_PATH)

X_nslkdd_test = nslkdd_test_df.drop(columns=["label"])
y_nslkdd_test = nslkdd_test_df["label"]

X_unsw_test = unsw_test_df.drop(columns=["label", "attack_cat"])
y_unsw_test = unsw_test_df["label"]

X_nslkdd_test_scaled = nslkdd_scaler.transform(X_nslkdd_test)
X_unsw_test_scaled = unsw_scaler.transform(X_unsw_test)

X_nslkdd_test_cnn = X_nslkdd_test_scaled.reshape(X_nslkdd_test_scaled.shape[0], X_nslkdd_test_scaled.shape[1], 1)
X_unsw_test_cnn = X_unsw_test_scaled.reshape(X_unsw_test_scaled.shape[0], X_unsw_test_scaled.shape[1], 1)

print("NSL-KDD CNN test shape:", X_nslkdd_test_cnn.shape)
print("UNSW-NB15 CNN test shape:", X_unsw_test_cnn.shape)

NSL-KDD CNN test shape: (22544, 41, 1)
UNSW-NB15 CNN test shape: (175341, 42, 1)


In [3]:
from pathlib import Path
import time
import os

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "datasets" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

NSLKDD_MODEL_PATH = MODEL_DIR / "nslkdd_cnn_baseline_model.h5"
UNSW_MODEL_PATH = MODEL_DIR / "unsw_cnn_baseline_model.h5"

NSLKDD_SCALER_PATH = MODEL_DIR / "nslkdd_scaler.pkl"
UNSW_SCALER_PATH = MODEL_DIR / "unsw_scaler.pkl"

NSLKDD_PRUNED_MODEL_PATH = MODEL_DIR / "nslkdd_cnn_pruned_model.h5"
UNSW_PRUNED_MODEL_PATH = MODEL_DIR / "unsw_cnn_pruned_model.h5"

PRUNING_RESULTS_PATH = RESULTS_DIR / "pruned_model_evaluation_results.csv"

print("NSL-KDD model exists:", NSLKDD_MODEL_PATH.exists())
print("UNSW-NB15 model exists:", UNSW_MODEL_PATH.exists())
print("NSL-KDD scaler exists:", NSLKDD_SCALER_PATH.exists())
print("UNSW-NB15 scaler exists:", UNSW_SCALER_PATH.exists())

NSL-KDD model exists: True
UNSW-NB15 model exists: True
NSL-KDD scaler exists: True
UNSW-NB15 scaler exists: True


In [2]:
try:
    import tensorflow_model_optimization as tfmot
    print("TensorFlow Model Optimization is installed.")
    print("Version:", tfmot.__version__)
except ImportError as e:
    print("TensorFlow Model Optimization is NOT installed.")
    print(e)

TensorFlow Model Optimization is NOT installed.
No module named 'tensorflow_model_optimization'


In [1]:
import sys
import tensorflow as tf

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)

Python executable: c:\Users\Admin\Desktop\Lightweight-IoT-IDS\thesis_env\Scripts\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
TensorFlow version: 2.21.0
